<a href="https://colab.research.google.com/github/bzhuang2-create/SURF---MC-Simulation-for-Educational-Research/blob/main/UI_Section.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Module: Effect of Density and Number of Atoms on RDF Shape

#fraction_list = [('B', 0.33), ('O', 0.33), ('S', 0.34)]
fraction_list = [('Ar', 1)]
#fraction_list = [('Ar', 0.5), ('C', 0.5)]
'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC(null_filler = None):
  clear_output()
  print("Running MC Simulation")

  Ns = np.linspace(N_slider.value[0], N_slider.value[1], atom_sweep_slider.value)
  for i in range(len(Ns)):
    Ns[i] = int(Ns[i])

  rho_lower = rho_slider.value[0]
  rho_upper = rho_slider.value[1]
  rho_sweep = rho_sweep_slider.value
  rhos_raw = np.linspace(rho_lower, rho_upper, rho_sweep) # mols / L
  rhos_simulation_units = []
  for i in range(len(rhos_raw)):
    rhos_simulation_units.append(rhos_raw[i] * 6.022e23 * 1000 * 1e-30)

  temperature = temp_slider.value

  niterations = iteration_slider.value[1]
  nwarmup = iteration_slider.value[0]

  fig2, axs2 = plt.subplots(len(rhos_simulation_units), len(Ns), figsize = (16, 12), sharex = True, sharey = True)
  fig2.suptitle('Average RDF', fontsize = 13)
  fig2.supxlabel('Distance (A)')
  fig2.supylabel('Radial Distribution Function')
  plt.tight_layout()

  fig3, axs3 = plt.subplots(len(rhos_simulation_units), len(Ns), figsize = (16, 12), sharex = True, sharey = True)
  fig3.suptitle('Species RDFs', fontsize = 13)
  fig3.supxlabel('Distance (A)')
  fig3.supylabel('Radial Distribution Function')
  plt.tight_layout()
  for i in (range(len(rhos_simulation_units))):
    for j in range(len(Ns)):
      rho = rhos_simulation_units[i]
      rho_normal_unit = rho / 6.022e23 / 1000 * 1e30
      N = int(Ns[j])

      Result = MC_main(rho, N, fraction_list, niterations, temperature,
                       energy = lennard_jones, energy_derivative = lennard_jones_derivative, params = LJ_avg_array, n_warmup = nwarmup)
      MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      ASE_RDF, manual_RDF, species_RDFs, mayer_total_RDF, mayer_species_RDFs = RDF

      ax_m = axs2[i, j]
      ax_m.plot(R_bins, manual_RDF, label = f"Actual: N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_m.plot(R_bins, mayer_total_RDF, label = f"Mayer: N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_m.legend()
      ax_m.set_title(f"N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_m.autoscale(enable = True, axis = 'both', tight = False)

      ax_s = axs3[i, j]
      for a in range (len(fraction_list)):
        for b in range(a, len(fraction_list)):
          index = cantor_pair(a, b)
          ax_s.plot(R_bins, species_RDFs[index], label = f"Actual: {fraction_list[a][0]}{fraction_list[b][0]}")
          ax_s.plot(R_bins, mayer_species_RDFs[index], label = f"Mayer: {fraction_list[a][0]}{fraction_list[b][0]}")

      ax_s.set_title(f"N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_s.autoscale(enable = True, axis = 'both', tight = False)
      ax_s.legend()

  plt.show()
  rebuild_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

N_slider = widgets.IntRangeSlider(
    value = [500, 2000],        # initial range [min_selected, max_selected]
    min = 300,               # minimum possible value
    max = 3000,              # maximum possible value
    step = 100,
    description = 'Atoms:',
    style = style,
    layout = Layout(width = width)
)

rho_slider = widgets.FloatRangeSlider(
    value = [55, 75],        # initial range [min_selected, max_selected]
    min = 1,               # minimum possible value
    max = 100,                  # maximum possible value
    step = 1,
    description = 'Density:',
    style = style,
    layout = Layout(width = width)
)

atom_sweep_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Atom Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

rho_sweep_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Density Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

temp_slider = widgets.IntSlider(
    value = 200,
    min = 0,
    max = 600,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

iteration_slider = widgets.IntRangeSlider(
    value = [0e5, 1e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_objects():
  display(button)
  display(N_slider)
  display(atom_sweep_slider)
  display(rho_slider)
  display(rho_sweep_slider)
  display(temp_slider)
  display(iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
button.on_click(run_MC)

rebuild_objects()

In [ ]:
# @title Module: Effect of Temperature and Density on RDF Shape

#fraction_list = [('B', 0.33), ('O', 0.33), ('S', 0.34)]
fraction_list = [('Ar', 1)]
#fraction_list = [('Ar', 0.5), ('C', 0.5)]
'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC(null_filler = None):
  clear_output()
  print("Running MC Simulation")

  N = atom_RDF_slider.value

  rho_lower = rho_RDF_slider.value[0]
  rho_upper = rho_RDF_slider.value[1]
  rho_sweep = rho_sweep_RDF_slider.value
  rhos_raw = np.linspace(rho_lower, rho_upper, rho_sweep) # mols / L
  rhos_simulation_units = []
  for i in range(len(rhos_raw)):
    rhos_simulation_units.append(rhos_raw[i] * 6.022e23 * 1000 * 1e-30)

  temp_lower = temp_RDF_slider.value[0]
  temp_upper = temp_RDF_slider.value[1]
  temp_sweep = temp_sweep_RDF_slider.value
  temps = np.linspace(temp_lower, temp_upper, temp_sweep)

  niterations = iteration_RDF_slider.value[1]
  nwarmup = iteration_RDF_slider.value[0]

  fig2, axs2 = plt.subplots(len(rhos_simulation_units), len(temps), figsize = (16, 12), sharex = True, sharey = True)
  fig2.suptitle('Average RDF', fontsize = 13)
  fig2.supxlabel('Distance (A)')
  fig2.supylabel('Radial Distribution Function')
  plt.tight_layout()

  fig3, axs3 = plt.subplots(len(rhos_simulation_units), len(temps), figsize = (16, 12), sharex = True, sharey = True)
  fig3.suptitle('Species RDFs', fontsize = 13)
  fig3.supxlabel('Distance (A)')
  fig3.supylabel('Radial Distribution Function')
  plt.tight_layout()
  for i in (range(len(rhos_simulation_units))):
    for j in range(len(temps)):
      rho = rhos_simulation_units[i]
      rho_normal_unit = rho / 6.022e23 / 1000 * 1e30
      temperature = temps[j]

      Result = MC_main(rho, N, fraction_list, niterations, temperature,
                       energy = lennard_jones, energy_derivative = lennard_jones_derivative, params = LJ_avg_array, n_warmup = nwarmup)
      MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      ASE_RDF, manual_RDF, species_RDFs, mayer_total_RDF, mayer_species_RDFs = RDF

      ax_m = axs2[i, j]
      ax_m.plot(R_bins, manual_RDF, label = f"Actual: rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_m.plot(R_bins, mayer_total_RDF, label = f"Mayer: rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_m.legend()
      ax_m.set_title(f"rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_m.autoscale(enable = True, axis = 'both', tight = False)

      ax_s = axs3[i, j]
      for a in range (len(fraction_list)):
        for b in range(a, len(fraction_list)):
          index = cantor_pair(a, b)
          ax_s.plot(R_bins, species_RDFs[index], label = f"Actual: {fraction_list[a][0]}{fraction_list[b][0]}")
          ax_s.plot(R_bins, mayer_species_RDFs[index], label = f"Mayer: {fraction_list[a][0]}{fraction_list[b][0]}")

      ax_s.set_title(f"rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_s.autoscale(enable = True, axis = 'both', tight = False)
      ax_s.legend()

  plt.show()
  rebuild_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

temp_RDF_slider = widgets.IntRangeSlider(
    value = [150, 350],        # initial range [min_selected, max_selected]
    min = 50,               # minimum possible value
    max = 800,              # maximum possible value
    step = 50,
    description = 'Temperature:',
    style = style,
    layout = Layout(width = width)
)

rho_RDF_slider = widgets.FloatRangeSlider(
    value = [55, 75],        # initial range [min_selected, max_selected]
    min = 1,               # minimum possible value
    max = 100,                  # maximum possible value
    step = 1,
    description = 'Density:',
    style = style,
    layout = Layout(width = width)
)

temp_sweep_RDF_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Temp Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

rho_sweep_RDF_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Density Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

atom_RDF_slider = widgets.IntSlider(
    value = 400,
    min = 200,
    max = 1000,
    step = 100,
    description = 'Atoms:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

iteration_RDF_slider = widgets.IntRangeSlider(
    value = [0e5, 1e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_objects():
  display(button_RDF)
  display(rho_RDF_slider)
  display(rho_sweep_RDF_slider)
  display(temp_sweep_RDF_slider)
  display(temp_RDF_slider)
  display(atom_RDF_slider)
  display(iteration_RDF_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
button_RDF = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
button_RDF.on_click(run_MC)

rebuild_objects()

In [ ]:
# @title Module: Energy Distribution as Simulation Progresses

energy_fraction_list = [('He', 1.0)]

x_values = []
y_values = []
energy_display = []
has_ran = False
boltzmann_params = 0
expon_params = 0
lowest_energy = 0

def get_energy_profile(index):
  if(index < len(energy_display)):
    return energy_display[index][0]
  elif (index < 0):
    return energy_display[0][0]
  else:
    return energy_display[len(energy_display) - 1][0]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_energy(null_filler = None):
  clear_output()
  print("Running MC Simulation - Energy Distribution Module")

  rho = 0.004
  no_atoms = 800
  temperature = energy_temp_slider.value
  niterations = energy_iteration_slider.value[1]
  nwarmup = energy_iteration_slider.value[0]

  Result = MC_main(rho, no_atoms, energy_fraction_list, niterations, temperature, params = LJ_avg_array, n_warmup = nwarmup, display_energy_profile = True, cube_start = True)
  MC, RDF, R_bins, no_bins, energy_profiles, heat_capacity, pressure, move_accepted_array = Result

  global energy_display
  energy_display = energy_profiles
  # energy_profiles: list of (histogram, bin_size, no_bins, cutoff_bin)

  first_profile = energy_display[0]
  frequency_values = first_profile[0]
  bin_size = first_profile[1]
  no_bins = first_profile[2]
  cutoff_bin = first_profile[3]

  bin_values = np.zeros(int(no_bins))
  bin_values[0] = -cutoff_bin * bin_size
  for i in range(1, no_bins):
    bin_values[i] = bin_values[i - 1] + bin_size

  global has_ran
  global x_values
  has_ran = True
  x_values = bin_values
  rebuild_energy_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

energy_temp_slider = widgets.IntSlider(
    value = 200,
    min = 0,
    max = 400,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

energy_iteration_slider = widgets.IntRangeSlider(
    value = [0e5, 4e5],
    min = 0e5,               # minimum possible value
    max = 4e5,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_energy_objects():
  display(energy_button)
  display(energy_temp_slider)
  display(energy_iteration_slider)

  if(has_ran):
    display(energy_interactive_plot)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
energy_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
energy_button.on_click(run_MC_energy)

slider = IntSlider(
    value = 0,
    min = 0,
    max = 800,
    step = 1,
    description = 'Move Me',
    continuous_update = True,
    style = style,
    layout = Layout(width = width)
)

def f(i = 0):
    x = x_values
    if(has_ran):
      y = get_energy_profile(i)
    else:
      return

    xnew = np.linspace(x[0], x[-1], 300)
    spline = make_interp_spline(x, y, k = 3)
    y_smooth = spline(xnew)

    plt.plot(xnew, y_smooth, label = 'data')
    plt.ylim(0, 1.0)
    plt.xlabel('Energy (eV)')
    plt.ylabel('Relative Frequency')
    plt.title('Energy Distribution')
    plt.legend()
    plt.show()

energy_interactive_plot = interactive(f, i = slider)
output = energy_interactive_plot.children[-1]
output.layout.height = '500px'
clear_output()

rebuild_energy_objects()

In [ ]:
# @title Module: Energy Distribution - Multiple Iterations

boltzmann_fraction_list = [('Ar', 1.0)]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_boltzmann(null_filler = None):
  clear_output()

  print("Running MC Simulation - Energy Distribution Module")

  N = 512
  rho = 0.004
  total = N
  iterations = 4

  temperature_low = boltzmann_temp_slider.value[0]
  temperature_high = boltzmann_temp_slider.value[1]
  temperatures = np.linspace(temperature_low, temperature_high, iterations)

  niterations = boltzmann_iteration_slider.value[1]
  nwarmup = boltzmann_iteration_slider.value[0]

  for temperature in temperatures:
    Result = MC_main(rho, total, boltzmann_fraction_list, niterations, temperature, params = LJ_avg_array, n_warmup = nwarmup, cube_start = True)
    MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result

    plt.hist(heat_capacity, bins = 30, density = True, label = f'T = {temperature:.1f} K', histtype = 'step')
  plt.xlabel('Energy (eV)')
  plt.ylabel('Relative Frequency')
  plt.title('Energy Distribution')
  plt.legend()
  plt.show()

  rebuild_boltzmann_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

boltzmann_temp_slider = widgets.IntRangeSlider(
    value = [250, 1500],
    min = 100,
    max = 1600,
    step = 50,
    description = 'Temperature:',
    style = style,
    layout = Layout(width = width)
)

boltzmann_iteration_slider = widgets.IntRangeSlider(
    value = [0.5e5, 4e5],
    min = 0.5e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_boltzmann_objects():
  display(boltzmann_button)
  display(boltzmann_temp_slider)
  display(boltzmann_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
boltzmann_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
boltzmann_button.on_click(run_MC_boltzmann)

rebuild_boltzmann_objects()

In [ ]:
# @title Module: Obtaining Normalized Heat Capacity

heat_capacity_fraction_list = [('Ar', 1.0)]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_heat_capacity(null_filler = None):
  clear_output()
  print("Running MC Simulation - Heat Capacity Module")

  trials = 5
  heat_capacity_array = np.zeros(trials)

  N = 256

  raw_density = 1202.05 # kg / m^3 (Argon)
  rho = raw_density * 1000 / 39.948 * 1e-10 * 1e-10 * 1e-10 * 6.022e23 # kg to grams, mols of Argon per gram, meters to A, mols to atoms
  total = N

  temperature = 100 # K
  niterations = heat_capacity_iteration_slider.value[1]
  nwarmup = heat_capacity_iteration_slider.value[0]

  accepted_moves = np.zeros(trials)
  accepted_moves_x = np.zeros(trials)
  for i in range(len(accepted_moves)):
    accepted_moves_x[i] = i

  x_temp = np.zeros(niterations - nwarmup - 1)
  for i in range(len(x_temp)):
    x_temp[i] = i

  for i in (range(trials)):
    Result = MC_main(rho, total, heat_capacity_fraction_list, niterations, temperature, params = LJ_avg_array, n_warmup = nwarmup, reference_start = True)
    MC, RDF, R_bins, no_bins, potential_energy_values, pressure, move_accepted_array = Result

    heat_capacity = np.var(potential_energy_values) / (total * temperature * temperature * kB * kB)
    heat_capacity_array[i] = heat_capacity
    plt.plot(x_temp, potential_energy_values)

    accepted_moves[i] = np.average(move_accepted_array)

  plt.xlabel('Iterations')
  plt.ylabel('Potential Energy (eV)')
  plt.title('Potential Energy vs Iteration')
  plt.show()
  heat_capacity_average = np.average(heat_capacity_array)
  heat_capacity_error = np.std(heat_capacity_array)

  print(f"The average normalized heat capacity is {heat_capacity_average} +/- {heat_capacity_error} 1 / atom")
  plt.hist(heat_capacity_array)
  plt.xlabel('Heat Capacity (1 / atom)')
  plt.ylabel('Frequency')
  plt.title('Heat Capacity Distribution')
  plt.show()

  plt.plot(accepted_moves_x, accepted_moves)
  plt.title('Rate of Accepted Moves')
  plt.show()

  rebuild_heat_capacity_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

heat_capacity_temp_slider = widgets.IntSlider(
    value = 200,
    min = 0,
    max = 400,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

heat_capacity_iteration_slider = widgets.IntRangeSlider(
    value = [1e5, 2e5],        # initial range [min_selected, max_selected]
    min = 0e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_heat_capacity_objects():
  display(heat_capacity_button)
  #display(heat_capacity_temp_slider)
  display(heat_capacity_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
heat_capacity_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
heat_capacity_button.on_click(run_MC_heat_capacity)

rebuild_heat_capacity_objects()

In [ ]:
# @title Module: Obtaining Pressure and Virial Coefficients

pressure_fraction_list = [('Ar', 1.0)]

'''
Virial expansion to fit for SciPy
density: the density of the system (mols / L)
B: the second virial coefficient  (A^3 / atom)
C: the third virial coefficient (A^6 / atom^2)

returns: the quantity P / kB * T
'''
def virial_equation_reduced(density, B, C):
  return density + (density ** 2) * B + (density ** 3) * C

'''
density: density (mols / L)
temperature: temperature (K)

Returns: the pressure (Pa)
'''
def ideal_gas_law(density, temperature):
  return density * R * temperature * 1000


'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_pressure(null_filler = None):
  clear_output()
  print("Running MC Simulation - Pressure Module")

  ref_model = LJRef(["argon"]) # note: this has to be hard-coded
  ideal_gas_law_array = []
  LJ_Ref_array = []

  iterations = 15
  no_atoms = 256

  RDF_pressure_array = np.zeros(iterations)
  virial_pressure_array = np.zeros(iterations)
  mayer_pressure_array = np.zeros(iterations)
  rhos_normal_units = np.geomspace(0.15, 50, iterations)

  temperature = pressure_temp_slider.value
  niterations = pressure_iteration_slider.value[1]
  nwarmup = pressure_iteration_slider.value[0]

  energy_arrays = []

  rhos = []
  for density in rhos_normal_units:
    rhos.append(density * 6.022e23 * 1000 * 1e-30) # convert to atoms / A^3
    ideal_gas_law_array.append(ideal_gas_law(density, temperature))

    # convert from mols / L to m^3 / mol
    molar_volume = (1 / density) / 1000
    LJ_Ref_array.append(reference_pressure(ref_model, molar_volume, temperature))


  for i in range(len(rhos)):
    rho = rhos[i]
    Result = MC_main(rho, no_atoms, pressure_fraction_list, niterations, temperature, n_warmup = nwarmup, reference_start = True,
                    params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
    MC, RDF, R_bins, no_bins, energy_array, pressure, move_accepted_array = Result

    RDF_pressure, virial_pressure, mayer_pressure = pressure
    RDF_pressure_array[i] = RDF_pressure
    virial_pressure_array[i] = virial_pressure
    mayer_pressure_array[i] = mayer_pressure
    energy_arrays.append(energy_array)

  result_pressure_RDF_normal_units = []
  result_pressure_virial_normal_units = []
  result_pressure_mayer_normal_units = []

  for i in range(len(RDF_pressure_array)):
    result_pressure_RDF_normal_units.append(RDF_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_virial_normal_units.append(virial_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_mayer_normal_units.append(mayer_pressure_array[i] * 1.602e-19 * 1e30)

  plt.scatter(rhos_normal_units, result_pressure_RDF_normal_units, label = 'RDF')
  plt.scatter(rhos_normal_units, result_pressure_virial_normal_units, label = 'Virial')
  plt.scatter(rhos_normal_units, result_pressure_mayer_normal_units, label = 'Mayer')
  plt.plot(rhos_normal_units, LJ_Ref_array, label = 'Reference LJ')
  plt.plot(rhos_normal_units, ideal_gas_law_array, label = 'Ideal Gas Law')
  plt.xlabel('Density (mol / L)')
  plt.ylabel('Pressure (Pa)')
  plt.title(f'Pressure vs Density - Data - T = {temperature}')
  plt.legend()
  plt.show()

  pressure_over_kBT_RDF = RDF_pressure_array / (kB * temperature)
  popt, pcov = curve_fit(virial_equation_reduced, rhos, pressure_over_kBT_RDF)
  B_RDF, C_RDF = popt
  perr = np.sqrt(np.diag(pcov))

  pressure_over_kBT_viral = virial_pressure_array / (kB * temperature)
  popt, pcov = curve_fit(virial_equation_reduced, rhos, pressure_over_kBT_viral)
  B_virial, C_virial = popt
  perr = np.sqrt(np.diag(pcov))

  pressure_over_kBT_mayer = mayer_pressure_array / (kB * temperature)
  popt, pcov = curve_fit(virial_equation_reduced, rhos, pressure_over_kBT_mayer)
  B_mayer, C_mayer = popt
  perr = np.sqrt(np.diag(pcov))

  fit_pressures_RDF = []
  fit_pressures_virial = []
  fit_pressures_mayer = []

  for i in range(len(rhos)):
    rho = rhos[i]
    fit_pressures_RDF.append(virial_equation_reduced(rho, B_RDF, C_RDF) * (kB * temperature))
    fit_pressures_virial.append(virial_equation_reduced(rho, B_virial, C_virial) * (kB * temperature))
    fit_pressures_mayer.append(virial_equation_reduced(rho, B_mayer, C_mayer) * (kB * temperature))

  pressures_normal_units_RDF = []
  pressures_normal_units_virial = []
  pressures_normal_units_mayer = []

  for i in range(len(fit_pressures_RDF)):
    pressures_normal_units_RDF.append(fit_pressures_RDF[i] * 1.602e-19 * 1e30) # pascals
    pressures_normal_units_virial.append(fit_pressures_virial[i] * 1.602e-19 * 1e30) # pascals
    pressures_normal_units_mayer.append(fit_pressures_mayer[i] * 1.602e-19 * 1e30) # pascals

  plt.scatter(rhos_normal_units, pressures_normal_units_RDF, label = 'RDF Fit')
  plt.scatter(rhos_normal_units, pressures_normal_units_virial, label = 'Virial Fit')
  plt.scatter(rhos_normal_units, pressures_normal_units_mayer, label = 'Mayer Fit')
  plt.plot(rhos_normal_units, LJ_Ref_array, label = 'Reference LJ')
  plt.plot(rhos_normal_units, ideal_gas_law_array, label = 'Ideal Gas Law')
  plt.xlabel('Density (mol / L)')
  plt.ylabel('Pressure (Pa)')
  plt.title(f'Pressure vs Density - Fit Equations - T = {temperature}')
  plt.legend()
  plt.show()

  print(f"RDF - The second virial coefficient is {B_RDF * 1e-30 * 6.022e23} +/- {perr[0]} m^3 / mol")
  print(f"RDF - The third vir ial coefficient is {C_RDF * 1e-60 * 6.022e23 * 6.022e23} +/- {perr[0]} m^6 / mol^2")
  print(f"Virial - The second virial coefficient is {B_virial * 1e-30 * 6.022e23} +/- {perr[0]} m^3 / mol")
  print(f"Virial - The third virial coefficient is {C_virial * 1e-60 * 6.022e23 * 6.022e23} +/- {perr[0]} m^6 / mol^2")
  print(f"Mayer - The second virial coefficient is {B_mayer * 1e-30 * 6.022e23} +/- {perr[0]} m^3 / mol")
  print(f"Mayer - The third virial coefficient is {C_mayer * 1e-60 * 6.022e23 * 6.022e23} +/- {perr[0]} m^6 / mol^2")

  rebuild_pressure_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

pressure_temp_slider = widgets.IntSlider(
    value = 200,
    min = 0.1,
    max = 600,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

pressure_iteration_slider = widgets.IntRangeSlider(
    value = [0.1e5, 1.5e5],        # initial range [min_selected, max_selected]
    min = 0e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

'''
Rebuilds all UI objects
'''
def rebuild_pressure_objects():
  display(pressure_button)
  display(pressure_temp_slider)
  display(pressure_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
pressure_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
pressure_button.on_click(run_MC_pressure)

rebuild_pressure_objects()

In [ ]:
# @title Module: Critical Point

pressure_fraction_list = [('Ar', 1.0)]


'''
Calculates the pressure according to ideal gas law
density: density (mols / L)
temperature: temperature (K)

Returns: the pressure in pascals
'''
def ideal_gas_law(density, temperature):
  return density * R * temperature * 1000


'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_pressure(null_filler = None):
  clear_output()
  print("Running MC Simulation - Pressure Module")

  ref_model = LJRef(["argon"]) # note: this has to be hard-coded
  ideal_gas_law_array = []
  LJ_Ref_array = []

  iterations = 10
  no_atoms = 256

  RDF_pressure_array = np.zeros(iterations)
  virial_pressure_array = np.zeros(iterations)
  mayer_pressure_array = np.zeros(iterations)
  molar_volume_normal_units = np.linspace(4e-9, 150, iterations) # m^3 / mol

  temperature = pressure_temp_slider.value
  niterations = pressure_iteration_slider.value[1]
  nwarmup = pressure_iteration_slider.value[0]

  energy_arrays = []
  molar_vols = []
  for spec_vol in molar_volume_normal_units:
    molar_vols.append(spec_vol / 6.022e23 * 1e30) # convert to A^3 / atom
    ideal_gas_law_array.append(ideal_gas_law(1 / spec_vol, temperature))
    LJ_Ref_array.append(reference_pressure(ref_model, spec_vol, temperature))

  for i in range(len(molar_vols)):
    molar_vol = molar_vols[i]
    rho = 1 / molar_vol
    rho_convert = rho
    Result = MC_main(rho_convert, no_atoms, pressure_fraction_list, niterations, temperature, n_warmup = nwarmup, reference_start = True,
                    params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
    MC, RDF, R_bins, no_bins, energy_array, pressure, move_accepted_array = Result

    RDF_pressure, virial_pressure, mayer_pressure = pressure
    RDF_pressure_array[i] = RDF_pressure
    virial_pressure_array[i] = virial_pressure
    mayer_pressure_array[i] = mayer_pressure
    energy_arrays.append(energy_array)

  result_pressure_RDF_normal_units = []
  result_pressure_virial_normal_units = []
  result_pressure_mayer_normal_units = []

  for i in range(len(RDF_pressure_array)):
    result_pressure_RDF_normal_units.append(RDF_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_virial_normal_units.append(virial_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_mayer_normal_units.append(mayer_pressure_array[i] * 1.602e-19 * 1e30)

  plt.scatter(molar_volume_normal_units, result_pressure_RDF_normal_units, label = 'RDF')
  plt.scatter(molar_volume_normal_units, result_pressure_virial_normal_units, label = 'Virial')
  plt.plot(molar_volume_normal_units, LJ_Ref_array, label = 'Reference LJ')
  plt.xlabel('Molar Volume (m^3 / mol)')
  plt.ylabel('Pressure (Pa)')
  plt.yscale('log')
  plt.title(f'Pressure vs Density - Data - T = {temperature}')
  plt.legend()
  plt.show()

  rebuild_pressure_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

pressure_temp_slider = widgets.IntSlider(
    value = 151,
    min = 0.1,
    max = 600,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

pressure_iteration_slider = widgets.IntRangeSlider(
    value = [0.1e5, 1.0e5],        # initial range [min_selected, max_selected]
    min = 0e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

'''
Rebuilds all UI objects
'''
def rebuild_pressure_objects():
  display(pressure_button)
  display(pressure_temp_slider)
  display(pressure_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
pressure_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
pressure_button.on_click(run_MC_pressure)

rebuild_pressure_objects()

In [ ]:
# @title Module: Pressure vs. Temperature

pressure_fraction_list = [('Ar', 1.0)]

'''
Calculates the pressure according to ideal gas law
density: density, in mols / L
temperature: temperature, in K

Returns: the pressure in pascals
'''
def ideal_gas_law(density, temperature):
  return density * R * temperature * 1000

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_pressure_temp(null_filler = None):
  ref_model = LJRef(["argon"]) # note: this has to be hard-coded
  ideal_gas_law_array = []
  LJ_Ref_array = []

  clear_output()
  print("Running MC Simulation - Pressure Module")

  iterations = 10
  samples = 1

  RDF_pressure_array = np.zeros(iterations)
  virial_pressure_array = np.zeros(iterations)
  mayer_pressure_array = np.zeros(iterations)
  reference_pressure_array = []

  no_atoms = 256
  temperatures = np.linspace(130, 900, iterations)

  # consider making this a slider
  niterations = pressure_temp_iteration_slider.value[1]
  nwarmup = pressure_temp_iteration_slider.value[0]

  energy_arrays = []
  temp_sample_array_RDF = []
  temp_sample_array_virial = []
  temp_sample_array_mayer = []

  density_raw = 44.44 # mol / L
  density = density_raw * 1000 * 1e-10 * 1e-10 * 1e-10 * 6.022e23

  for temperature in temperatures:
    ideal_gas_law_array.append(ideal_gas_law(density_raw, temperature))

    molar_vol_raw = 1 / (density_raw * 1000)

    reference_pressure_array.append(reference_pressure(ref_model, molar_vol_raw, temperature))

  for i in range(len(temperatures)):
    for j in range(samples):
      temp = temperatures[i]
      Result = MC_main(density, no_atoms, pressure_fraction_list, niterations, temp, n_warmup = nwarmup, reference_start = True,
                       params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
      MC, RDF, R_bins, no_bins, energy_array, pressure, move_accepted_array = Result

      RDF_pressure_sample, virial_pressure_sample, mayer_pressure_sample = pressure
      temp_sample_array_RDF.append(RDF_pressure_sample)
      temp_sample_array_virial.append(virial_pressure_sample)
      temp_sample_array_mayer.append(mayer_pressure_sample)

    RDF_pressure = np.average(temp_sample_array_RDF)
    virial_pressure = np.average(temp_sample_array_virial)
    mayer_pressure = np.average(temp_sample_array_mayer)
    temp_sample_array_RDF.clear()
    temp_sample_array_virial.clear()
    temp_sample_array_mayer.clear()

    RDF_pressure_array[i] = RDF_pressure
    virial_pressure_array[i] = virial_pressure
    mayer_pressure_array[i] = mayer_pressure
    energy_arrays.append(energy_array)

  result_pressure_RDF_normal_units = []
  result_pressure_virial_normal_units = []
  result_pressure_mayer_normal_units = []

  for i in range(len(RDF_pressure_array)):
    result_pressure_RDF_normal_units.append(RDF_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_virial_normal_units.append(virial_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_mayer_normal_units.append(mayer_pressure_array[i] * 1.602e-19 * 1e30)

  plt.scatter(temperatures, result_pressure_RDF_normal_units, label = 'RDF')
  plt.scatter(temperatures, result_pressure_virial_normal_units, label = 'Virial')
  plt.scatter(temperatures, result_pressure_mayer_normal_units, label = 'Mayer')
  plt.plot(temperatures, ideal_gas_law_array, label = 'Ideal Gas Law')
  plt.plot(temperatures, reference_pressure_array, label = 'Reference LJ')
  plt.xlabel('Temperature (K)')
  plt.ylabel('Pressure (Pa)')
  plt.title('Pressure vs Temperature - Data')
  plt.legend()
  plt.show()

  iterations_array = np.zeros(niterations - nwarmup - 1)
  for i in range(len(iterations_array)):
    iterations_array[i] = i

  for i in range(len(energy_arrays)):
    energy_history = energy_arrays[i]
    plt.plot(iterations_array, energy_history, label = f'{i}')
  plt.xlabel('Iterations')
  plt.ylabel('Potential Energy (eV)')
  plt.title('Potential Energy vs Iteration')
  plt.legend()
  plt.show()

  rebuild_pressure_temp_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

pressure_temp_iteration_slider = widgets.IntRangeSlider(
    value = [1e5, 2e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

'''
Rebuilds all UI objects
'''
def rebuild_pressure_temp_objects():
  display(pressure_temp_button)
  display(pressure_temp_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
pressure_temp_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
pressure_temp_button.on_click(run_MC_pressure_temp)

rebuild_pressure_temp_objects()

In [ ]:
# @title Module: Signal to Noise Ratio as Number of Atoms Increases

noise_fraction_list = [('Ar', 1.0)]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_noise(null_filler = None):
  clear_output()
  print("Running MC Simulation - Signal to Noise Module")

  # how many different N values will be tested
  iterations = 10
  samples = 1
  # how many MC simulations will be run for each N value to determine the SNR ratio

  final_average_RDF_array = []

  N_low = noise_N_slider.value[0]
  N_high = noise_N_slider.value[1]
  Ns = np.linspace(N_low, N_high, iterations)

  rho = 0.015
  temperature = 100
  niterations = noise_iteration_slider.value[1]
  nwarmup = noise_iteration_slider.value[0]

  signal_to_noise_array = np.zeros(iterations)
  temp_array = np.zeros(samples)

  for i in range(len(Ns)):
    SNR_accumulator = 0

    for j in range(samples):
      N = int(Ns[i])
      Result = MC_main(rho, N, noise_fraction_list, niterations, temperature, n_warmup = nwarmup,
                       params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
      MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      ASE_RDF, manual_RDF, _, _, _ = RDF
      final_average_RDF_array.append(manual_RDF)

    for k in range(no_bins):
      for h in range(samples):
        temp_array[h] = final_average_RDF_array[h][k] # takes the kth bin of every RDF for this iteration

      if(0 in temp_array):
        continue

      signal = np.average(temp_array)
      noise = np.std(temp_array)
      if(noise <= 0): noise = 1e-4

      SNR_accumulator += signal / noise
      # raw ratio because log was giving negative numbers

    signal_to_noise_array[i] = SNR_accumulator / samples
    final_average_RDF_array.clear()

  one_over_sqrt_N = 1 / np.sqrt(Ns)

  plt.plot(one_over_sqrt_N, signal_to_noise_array)
  plt.xlabel('1 / sqrt(N)')
  plt.ylabel('Signal to Noise Ratio')
  plt.title('Signal to Noise Ratio vs 1 / sqrt(N)')
  plt.show()

  rebuild_noise_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)


noise_N_slider = widgets.IntRangeSlider(
    value = [256, 1024],        # initial range [min_selected, max_selected]
    min = 128,               # minimum possible value
    max = 2048,              # maximum possible value
    step = 32,
    description = 'Atoms:',
    style = style,
    layout = Layout(width = width)
)

noise_iteration_slider = widgets.IntRangeSlider(
    value = [1e5, 1e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_noise_objects():
  display(noise_button)
  display(noise_N_slider)
  display(noise_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
noise_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
noise_button.on_click(run_MC_noise)

rebuild_noise_objects()